# ✅ Solution: Debug Your Synthetic Dataset

This notebook contains our walkthrough of the challenge from Video 7. We identified three issues in the synthetic credit dataset and propose a concrete fix for each one. Compare it against your own analysis before reading on.

## 1. Load the Datasets

In [7]:
import pandas as pd

original_df  = pd.read_csv('../data/credit_original.csv')
synthetic_df = pd.read_csv('../data/credit_synthetic.csv')

print(f'Original : {original_df.shape[0]:,} rows × {original_df.shape[1]} columns')
print(f'Synthetic: {synthetic_df.shape[0]:,} rows × {synthetic_df.shape[1]} columns')

Original : 1,000 rows × 7 columns
Synthetic: 1,000 rows × 7 columns


## 2. Profiling Report

In [8]:
from ydata_profiling import ProfileReport

original_profile  = ProfileReport(original_df,  title='Original',  minimal=True)
synthetic_profile = ProfileReport(synthetic_df, title='Synthetic', minimal=True)

comparison = original_profile.compare(synthetic_profile)
comparison.to_file('credit_profiling_comparison.html')
print('Profiling report saved → credit_profiling_comparison.html')
comparison

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|████████████████████████████████████████| 7/7 [00:00<00:00, 125470.63it/s]


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|████████████████████████████████████████| 7/7 [00:00<00:00, 184654.89it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling report saved → credit_profiling_comparison.html


## 3. Quality Report

In [9]:
from ydata.dataset import Dataset
from ydata.metadata import Metadata
from ydata.report import SyntheticDataProfile

original_dataset  = Dataset(original_df)
synthetic_dataset = Dataset(synthetic_df)
metadata          = Metadata(original_dataset)

profile = SyntheticDataProfile(
    original_dataset,
    synthetic_dataset,
    metadata=metadata,
    target='defaulted',
)

profile.generate_report(output_path='credit_quality_report.pdf')
print('Quality report saved → credit_quality_report.pdf')

INFO: 2026-08-25 20:48:43,126 Using device of type: cpu


Done:  80%|████████████████████████████       | 4/5 [00:00<00:00, 163.59step/s]

INFO: 2026-08-25 20:48:43,947 [PROFILEREPORT] - Starting metrics calculation.
INFO: 2026-08-25 20:48:43,951 [PROFILEREPORT] - Synthetic data quality report selected target variable: defaulted
INFO: 2026-08-25 20:48:43,953 [PROFILEREPORT] - preparing data format.
INFO: 2026-08-25 20:48:43,959 [PROFILEREPORT] - Preparing the data for metrics calculation
INFO: 2026-08-25 20:48:43,968 [PROFILEREPORT] - Calculating privacy metrics.
INFO: 2026-08-25 20:48:43,969 [PROFILEREPORT] - Calculating metric [Exact Matches].
INFO: 2026-08-25 20:48:43,971 [PROFILEREPORT] - Metric [Exact Matches] took 0.00s.
INFO: 2026-08-25 20:48:43,972 [PROFILEREPORT] - Calculating metric [Identifiability Risk].
INFO: 2026-08-25 20:48:43,976 [PROFILEREPORT] - Metric [Identifiability Risk] took 0.00s.
INFO: 2026-08-25 20:48:43,976 [PROFILEREPORT] - Calculating fidelity metrics.
INFO: 2026-08-25 20:48:43,978 [PROFILEREPORT] - Calculating metric [Correlation Similarity].
INFO: 2026-08-25 20:48:43,996 [PROFILEREPORT] - Me

INFO: 2026-08-25 20:48:44,191 [PROFILEREPORT] - Metric [Mutual Information] took 0.17s.
INFO: 2026-08-25 20:48:44,192 [PROFILEREPORT] - Calculating metric [Dimensionality Reduction].
INFO: 2026-08-25 20:48:44,212 [PROFILEREPORT] - Metric [Dimensionality Reduction] took 0.02s.
INFO: 2026-08-25 20:48:44,213 [PROFILEREPORT] - Calculating metric [Synthetic Classifier].
INFO: 2026-08-25 20:48:44,280 [PROFILEREPORT] - Metric [Synthetic Classifier] took 0.07s.
INFO: 2026-08-25 20:48:44,281 [PROFILEREPORT] - Calculating utility metrics.
INFO: 2026-08-25 20:48:44,283 [PROFILEREPORT] - Calculating metric [QScore].
INFO: 2026-08-25 20:48:44,311 [PROFILEREPORT] - Metric [QScore] took 0.03s.
INFO: 2026-08-25 20:48:44,311 [PROFILEREPORT] - Calculating metric [Train on Synthetic Test on Real].
ERROR: 2026-08-25 20:48:44,634 boolean index did not match indexed array along axis 0; size of axis is 1 but size of corresponding boolean axis is 2
Traceback (most recent call last):
  File "/Users/fabianaclem

# Real data records: 890
# Synthetic data records generated: 22
# Columns: 7
Fidelity Score: Very Good
Utility Score: Poor
Privacy Score: Excellent


Correlation Similarity: 0.96
Distance Distribution: 0.88
Synthetic Classifier: 1.00
Mutual Information: 0.94
Missing Values Similarity: 1.00
Mean Similarity: 0.98
Std. Dev. Similarity: 0.98
Median Similarity: 0.98
Q25% Similarity: 0.97
Q75% Similarity: 0.97
Kolmogorov-Smirnov Test: 0.85
Total Variation Distance: 0.81
Category Coverage: 0.84
Missing Category Coverage: 0.02
Range Coverage: 0.63


QScore: 0.37
Feature Importance: None


Exact Matches: 0.0
Identifiability Risk: 0.02
Quality report saved → credit_quality_report.pdf


## 4. Solution

### Question 1 — Does the synthetic dataset present major quality issues?

Yes — three clear issues, each in a different dimension.

#### Issue 1 — Class imbalance in `defaulted` (Utility)

The target column has a severe class imbalance problem in the synthetic data. The original dataset has ~16% defaults; the synthetic version has ~52%. A model trained on this synthetic data would learn a completely different prior for the target variable.

In [10]:
print('Target distribution — defaulted:')
dist = pd.concat([
    original_df['defaulted'].value_counts(normalize=True).rename('original %'),
    synthetic_df['defaulted'].value_counts(normalize=True).rename('synthetic %'),
], axis=1).mul(100).round(1)
print(dist.to_string())

Target distribution — defaulted:
           original %  synthetic %
defaulted                         
0                84.0         96.0
1                16.0          4.0


#### Fix 1 — Class imbalance: conditional training

A post-hoc alternative is to undersample the overrepresented class in the synthetic output. 

In [14]:
# Option 1
# Post-hoc fix: undersample the majority class to restore the original ratio
# The simplest method
target_rate = (original_df['defaulted'] == 1).mean()
n_minority  = (synthetic_df['defaulted'] == 1).sum()
n_majority_target = int(n_minority / target_rate * (1 - target_rate))

majority = synthetic_df[synthetic_df['defaulted'] == 0].sample(n_majority_target, random_state=42)
minority = synthetic_df[synthetic_df['defaulted'] == 1]
rebalanced_df = pd.concat([majority, minority]).sample(frac=1, random_state=42).reset_index(drop=True)

print('After rebalancing:')
print(rebalanced_df.shape)
print(rebalanced_df['defaulted'].value_counts(normalize=True).mul(100).round(1).to_string())

After rebalancing:
(250, 7)
defaulted
0    84.0
1    16.0


In [18]:
# Option 2
# Conditionally train a new synthetic data generator: Force the synthesizer to learn the underrepresented distribution
# The best if you want to be precise about the quality of your distributions
from ydata.dataset import Dataset
from ydata.metadata import Metadata
from ydata.synthesizers import RegularSynthesizer

dataset = Dataset(original_df)
metadata = Metadata(dataset)

synth = RegularSynthesizer()
# condition the synthesizer to the values of the column `defaulted`
synth.fit(dataset, metadata=metadata, condition_on=['defaulted'])

synthetic_sample = synth.sample(len(dataset))

print(synthetic_sample.to_pandas()['defaulted'].value_counts(normalize=True).mul(100).round(1).to_string())

Done:  80%|████████████████████████████       | 4/5 [00:00<00:00, 109.46step/s]

Validate the segmentation strategy.
INFO: 2026-08-25 20:50:41,596 [SYNTHESIZER] - Number columns considered for synth: 7


INFO: 2026-08-25 20:50:42,482 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-25 20:50:42,486 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-25 20:50:42,489 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-25 20:50:42,490 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
INFO: 2026-08-25 20:50:42,562 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-25 20:50:42,580 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-25 20:50:42,581 [SYNTHESIZER] - Numerical clipping
INFO: 2026-08-25 20:50:42,586 let write into the connector None
INFO: 2026-08-25 20:50:42,586 using kwargs {}
defaulted
0    85.6
1    14.4


#### Issue 2 — Broken relationship between `age` and `age_group` (Fidelity)

`age_group` is a deterministic function of `age` — it is a binned version of the same variable. Generative models are probabilistic and do not understand deterministic rules, so they learn the relationship approximately and occasionally violate it. The fix is to remove `age_group` from the training data and recompute it as a calculated feature after generation.

In [19]:
# Show records where age_group does not match age in the synthetic data
def expected_age_group(age):
    if age < 30:   return 'young'
    elif age < 50: return 'middle'
    else:          return 'senior'

synthetic_df['expected_age_group'] = synthetic_df['age'].apply(expected_age_group)
violations = synthetic_df[synthetic_df['age_group'] != synthetic_df['expected_age_group']]

print(f'Records where age_group does not match age: {len(violations):,} '
      f'({len(violations)/len(synthetic_df)*100:.1f}%)')
violations[['age', 'age_group', 'expected_age_group']].head(10)

Records where age_group does not match age: 1,000 (100.0%)


,age,age_group,expected_age_group
0,22,22-30,young
1,41,31-45,middle
2,27,22-30,young
3,37,22-30,middle
4,39,31-45,middle
5,39,31-45,middle
6,36,61+,middle
7,30,22-30,middle
8,68,61+,senior
9,63,61+,senior


#### Fix 2 — Broken relationship: treat `age_group` as a calculated feature

Remove `age_group` from the training data entirely and recompute it deterministically from `age` after generation.

In [20]:
def compute_age_group(age):
    return pd.cut(
        age,
        bins=[0, 29, 49, 120],
        labels=['young', 'middle', 'senior']
    ).astype(str).values

calculated_features = [
    {
        'calculated_features': 'age_group',
        'function': compute_age_group,
        'calculated_from': ['age'],
    }
]

# The synthesizer trains without age_group and enforces the rule at generation time
#synth.fit(X=dataset, metadata=metadata, calculated_features=calculated_features)

#### Issue 3 — Quasi-identifiers: `age`, `income`, `zip_prefix` (Privacy)

The dataset contains three quasi-identifiers. Age, income and zip_prefix combine can re-identify one individual. Let's ensure that is not possible by masking zip_prefix during the process of synthesis.

In [21]:
# zip_prefix carries the highest re-identification risk — show its uniqueness
n_unique_zip  = original_df['zip_prefix'].nunique()
n_total       = len(original_df)
print(f'zip_prefix: {n_unique_zip} unique values in {n_total:,} records '
      f'({n_unique_zip/n_total*100:.1f}% uniqueness)')

zip_prefix: 10 unique values in 1,000 records (1.0% uniqueness)


#### Fix 3 — Quasi-identifiers: anonymize `zip_prefix`

`zip_prefix` carries the highest re-identification risk due to its uniqueness. We anonymize it before training using a faker method. The privacy score for `age` and `income` was within the acceptable range, so we leave them as-is — dropping `age` in favour of `age_group` alone is an option if the privacy score worsens after retraining.

In [28]:
anonymize = {
    # Regex as a string is deduced automatically as AnonymizerType.REGEX
    'zip_prefix': {
        "type": "regex",
        # regex anonymizer's specific required parameter
        "regex": r'[0-9]{4}-[A-Z]{5}',
    }
}

# synth.fit(X=dataset, metadata=metadata, anonymize=anonymize)

### Combining all together

In [29]:
from ydata.dataset import Dataset
from ydata.metadata import Metadata
from ydata.synthesizers import RegularSynthesizer

dataset = Dataset(original_df)
metadata = Metadata(dataset)

synth.fit(X=dataset, 
          metadata=metadata, 
          condition_on=['defaulted'],
          anonymize=anonymize, 
          calculated_features=calculated_features)

synthetic_sample=synth.sample(len(dataset))

Done:  80%|████████████████████████████       | 4/5 [00:00<00:00, 107.58step/s]

Validate the segmentation strategy.
INFO: 2026-08-25 20:54:07,785 [SYNTHESIZER] - Number columns considered for synth: 6


INFO: 2026-08-25 20:54:08,108 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-25 20:54:08,111 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-25 20:54:08,114 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-25 20:54:08,114 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
INFO: 2026-08-25 20:54:08,174 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-25 20:54:08,187 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-25 20:54:08,188 [SYNTHESIZER] - Numerical clipping
INFO: 2026-08-25 20:54:08,195 let write into the connector None
INFO: 2026-08-25 20:54:08,195 using kwargs {}


In [33]:
synthetic_sample.head()

,age,income,credit_score,defaulted,occupation,zip_prefix,age_group
0,49,26408.65,587,0,Teacher,<NA>,middle
1,30,115010.10,672,0,Manager,<NA>,middle
2,54,37005.85,595,1,Engineer,<NA>,senior
3,22,38895.94,576,1,Analyst,<NA>,young
4,49,76682.62,771,0,Analyst,<NA>,middle
